<a href="https://colab.research.google.com/github/stephanie465337/Data-Science-Portfolio-C21/blob/main/NotebookLM/Module%202/5c_Logistic_Regression_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic regression

## Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn.model_selection as model_selection
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc


## Read In & Understand Data
We will be looking at the Pima Indians Diabetes dataset that can be used to predict whether or not someone has diabetes based on a set of a predictors. Information about this dataset is provided [here](https://www.kaggle.com/uciml/pima-indians-diabetes-database).

In [ ]:
url = "http://ddc-datascience.s3.amazonaws.com/pima-indians-diabetes.csv"
url


In [ ]:
!curl -I {url}


Notice ...
- 200 response
- content type of text/csv
- length of about 24kb

In [ ]:
!curl -s {url} | head


Notice ...
- comma separated values
- a mix of floats and ints
- no cloumn headings


## Read in the data



Notice that the data set does not have column names.  So, we will create our own and include them when we create the data frame.


In [ ]:
col_names = ['pregnant', 'glucose', 'bp', 'skin', 'insulin', 'bmi', 'pedigree', 'age', 'label']
pima = pd.read_csv(
  url,
  header=None,
  names=col_names,
)
pima


In [ ]:
pima.head()

In [ ]:
pima.shape

In [ ]:
pima.info()

In [ ]:
pima.describe().transpose()

In [ ]:
pima['label'].value_counts( dropna = False)


In [ ]:
X = pima.drop('label',axis=1).copy()
y = pima['label'].copy()

## EDA

### Correlation matrix plot

In [ ]:
# Checking correlations between predictors
plt.figure(figsize=(10,10))
correlation_matrix = X.corr().round(2)
sns.heatmap(
  data=correlation_matrix,
  cmap='coolwarm',
  annot=True,
  vmin=-1,
  vmax=1
)
plt.show()


### Feature distributions

In [ ]:
# Look at the distribution of a few predictors for different labels
predictors = ['glucose', 'bmi', 'bp', 'pregnant']

fig, axes = plt.subplots(1,4, figsize= (22,4))
for i,pred in enumerate(predictors):
  sns.kdeplot(pima[pred][y==1], label = 'Yes', ax = axes[i])
  sns.kdeplot(pima[pred][y==0], label = 'No', ax = axes[i])
  axes[i].legend()
plt.show()


## Fit Logistic Model

### Cross validation

In [ ]:
numLoops = 50
predict_accuracy = np.zeros(numLoops)
predict_f1 = np.zeros(numLoops)

for idx in range(numLoops):
  # Train/test split
  X_train, X_test, y_train, y_test = model_selection.train_test_split( X, y, test_size=0.2 )

  # Create model
  logreg = LogisticRegression( max_iter=1000 )

  # Fit ( train ) model
  logreg.fit( X_train, y_train )

  # Predict
  y_pred = logreg.predict( X_test )

  # Calculate and record performance metrics
  predict_accuracy[idx] = metrics.accuracy_score(y_test,y_pred)
  predict_f1[idx] = metrics.f1_score(y_test,y_pred)

print(f"Mean Accuracy: {predict_accuracy.mean()*100:.1f}%")
print(f"F1 Score: {predict_f1.mean()*100:.1f}%")

In [ ]:
for _ in X_train, X_test, y_train, y_test:
  print(_.shape)

### Confusion matrix

In [ ]:
# Let's look at the confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

We often predict that someone does not have diabetes even if they do. What can we do to combat this?

### Balance classes

In [ ]:
n = pima["label"].value_counts().min()
n

In [ ]:
# Let's balance our classes
pima_balanced = pima.groupby('label').sample(n = n, replace = False)

In [ ]:
pima_balanced['label'].value_counts( dropna = False)

In [ ]:
X = pima_balanced.drop('label',axis=1).copy()
y = pima_balanced['label'].copy()

### Cross validation

In [ ]:
numLoops = 50
predict_accuracy = np.zeros(numLoops)
predict_f1 = np.zeros(numLoops)
for idx in range(numLoops):
  # Train/test split
  X_train, X_test, y_train, y_test = model_selection.train_test_split(X,y,test_size=0.2)

  # Create model
  logreg = LogisticRegression(max_iter=1000)

  # Fit ( train ) model
  logreg.fit(X_train,y_train)

  # Predict
  y_pred = logreg.predict(X_test)

  # Calculate and record performance metrics
  predict_accuracy[idx] = metrics.accuracy_score(y_test,y_pred)
  predict_f1[idx] = metrics.f1_score(y_test,y_pred)

print(f"Mean Accuracy: {predict_accuracy.mean()*100:.1f}%")
print(f"F1 Score: {predict_f1.mean()*100:.1f}%")

### Confusion matrix

Let's look at the confusion matrix using the last validation run


In [ ]:
cm = confusion_matrix( y_test, y_pred )
plt.figure(figsize=(8,8))
disp = ConfusionMatrixDisplay( confusion_matrix = cm )
disp.plot()
plt.show()


How does the model make its classifications?

In [ ]:
y_pred_proba = logreg.predict_proba(X_test)[:,1]
print(y_pred_proba[0:15]*100)

Pair up preditions with probabilities.  The default threshold is 50%.

In [ ]:
sorted( zip( y_pred, y_pred_proba[0:15]*100))


## ROC-AUC



A receiver operating characteristic ( ROC ) and area under the curve ( AUC ) plot illustrates the performance of a binary classifier model at varying threshold values.  It also enables us to pick an optimal threshold for classification.






### Compute ROC curve and AUC


In [ ]:
fpr, tpr, thresholds = roc_curve( y_test, y_pred_proba )
roc_auc = auc(fpr, tpr)


In [ ]:
roc_auc

### Plot ROC curve


In [ ]:
plt.figure()
plt.plot(
  fpr,
  tpr,
  color = 'blue',
  label = f'ROC curve (area = {roc_auc:.2f})',
)
plt.plot(
  [0, 1],
  [0, 1],
  color='red',
  linestyle='--',
)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()


In [ ]:
np.array([thresholds[:10], fpr[:10], tpr[:10], (tpr - fpr)[:10]]).T

In [ ]:
(tpr - fpr).max()

In [ ]:
idx = (tpr - fpr).argmax()
idx


In [ ]:
(tpr - fpr)[idx]

In [ ]:
# Find the optimal threshold
optimal_idx = np.argmax(tpr - fpr)  # Point closest to top-left corner
optimal_threshold = thresholds[optimal_idx]

print(f'Optimal threshold: {optimal_threshold}')


### Calculate predictions using ROC-AUC threshold


In [ ]:
y_pred = ( y_pred_proba >= optimal_threshold ).astype(int)
y_pred = ( y_pred_proba >= 0.15 ).astype(int)

y_pred

### Confusion matrix

In [ ]:
# Let's look at the confusion matrix
cm = confusion_matrix( y_test, y_pred )
plt.figure(figsize=(8,8))
disp = ConfusionMatrixDisplay( confusion_matrix = cm )
disp.plot()
plt.show() ;


### Calculate performance metrics

In [ ]:
predict_accuracy = metrics.accuracy_score( y_test, y_pred )
predict_f1 = metrics.f1_score( y_test, y_pred )

print(f"Mean Accuracy: {predict_accuracy*100:.1f}%")
print(f"F1 Score: {predict_f1*100:.1f}%")

## References

- [ROC-AUC]( https://en.wikipedia.org/wiki/Receiver_operating_characteristic )